# Step 9-10: Specialist Fine-tuning and Final Comparison

This notebook fine-tunes a specialist text classifier on the Claude-labeled train set and compares it against GenAI models on the holdout.

In [1]:
# If needed:
# !pip install -U pandas numpy scikit-learn torch transformers datasets accelerate evaluate sentencepiece tiktoken protobuf


In [2]:
from __future__ import annotations

import json
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

ROOT = Path.cwd()
TRAIN_TEXT_CSV = ROOT / "data" / "mortgage_train_15000.csv"
TRAIN_LABEL_CSV = ROOT / "gen_ai_labeling" / "outputs" / "claude_code__sonnet-4-6_train" / "labels.csv"
HOLDOUT_TEXT_CSV = ROOT / "data" / "mortgage_holdout_1000.csv"
HOLDOUT_TRUTH_CSV = ROOT / "data" / "final_human_consensus_holdout.csv"
GEN_AI_OUTPUT_DIR = ROOT / "gen_ai_labeling" / "outputs"
ARTIFACT_DIR = ROOT / "specialist_model_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["improper_charges", "improper_process", "deceptive_discriminatory"]
LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prefer CUDA, then Apple Silicon MPS, then CPU. On an M1/M2 Mac the MPS
# backend gives a 5-10x speedup over CPU for transformer fine-tuning.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [3]:
def normalize_label(label: str) -> str:
    x = str(label).strip()
    if x == "deceptive_and_discriminatory_practices":
        return "deceptive_discriminatory"
    return x

df_text = pd.read_csv(TRAIN_TEXT_CSV)[["complaint_id", "complaint_what_happened"]]
df_lab = pd.read_csv(TRAIN_LABEL_CSV)[["complaint_id", "complaint_category"]]
df_lab["complaint_category"] = df_lab["complaint_category"].map(normalize_label)
df_lab = df_lab[df_lab["complaint_category"].isin(LABELS)].copy()

df = df_text.merge(df_lab, on="complaint_id", how="inner")
df["text"] = df["complaint_what_happened"].fillna("").astype(str).str.strip()
df = df[df["text"].str.len() > 0].copy()
df["label"] = df["complaint_category"].map(LABEL2ID)
df = df[["complaint_id", "text", "complaint_category", "label"]].reset_index(drop=True)

print("Training rows:", len(df))
print(df["complaint_category"].value_counts())

Training rows: 14803
complaint_category
improper_process            5862
deceptive_discriminatory    5711
improper_charges            3230
Name: count, dtype: int64


In [4]:
# Use only train for model selection; holdout stays untouched until the end.
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("train:", len(train_df), "val:", len(val_df))

train: 12582 val: 2221


In [5]:
@dataclass
class RunConfig:
    name: str
    model_name: str
    learning_rate: float
    num_train_epochs: int
    train_batch_size: int
    weight_decay: float
    max_length: int

# Single strong recipe, picked to clear macro-F1 >= 0.80 with one pass.
# Token-length analysis showed p50=288 tokens, so max_length=384 keeps ~66%
# of texts un-truncated while staying ~33% faster than 512.
# A second config is left commented as a fallback if the first underperforms.
CONFIGS = [
    RunConfig("roberta_base_lr2e5_ep4_len384", "roberta-base", 2e-5, 4, 16, 0.01, 384),
    # RunConfig("roberta_base_lr15e5_ep5_len512", "roberta-base", 1.5e-5, 5, 8, 0.01, 512),
]
print("Will run", len(CONFIGS), "configuration(s)")

Will run 1 configuration(s)


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

def make_dataset(df_in: pd.DataFrame, tokenizer, max_length: int):
    ds = Dataset.from_pandas(df_in[["text", "label"]], preserve_index=False)
    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)
    return ds.map(tok, batched=True)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Balanced class weights help macro-F1 when one class underperforms
# (improper_charges is ~22% of the train set vs ~39% / ~39% for the others).
label_counts = train_df["label"].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df) / (len(LABELS) * label_counts[i]) for i in range(len(LABELS))],
    dtype=torch.float,
)

# Mixed precision: only on CUDA. MPS support for fp16 in Trainer is still
# rough as of torch 2.x and can produce NaNs; bf16 is CUDA/CPU only.
USE_FP16 = (DEVICE == "cuda")
USE_BF16 = False

all_results = []
best = None

for cfg in CONFIGS:
    print(f"\n=== Training: {cfg.name} ({cfg.model_name}) on {DEVICE} ===")
    t0 = time.time()
    try:
        tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(
            cfg.model_name,
            num_labels=len(LABELS),
            id2label=ID2LABEL,
            label2id=LABEL2ID,
        )

        train_ds = make_dataset(train_df, tokenizer, cfg.max_length)
        val_ds = make_dataset(val_df, tokenizer, cfg.max_length)

        run_dir = ARTIFACT_DIR / "runs" / cfg.name
        run_dir.mkdir(parents=True, exist_ok=True)

        args = TrainingArguments(
            output_dir=str(run_dir),
            learning_rate=cfg.learning_rate,
            per_device_train_batch_size=cfg.train_batch_size,
            per_device_eval_batch_size=max(32, cfg.train_batch_size * 2),
            gradient_accumulation_steps=1,
            num_train_epochs=cfg.num_train_epochs,
            weight_decay=cfg.weight_decay,
            warmup_ratio=0.06,
            lr_scheduler_type="linear",
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            fp16=USE_FP16,
            bf16=USE_BF16,
            dataloader_num_workers=2,
            dataloader_pin_memory=(DEVICE == "cuda"),
            logging_steps=50,
            report_to="none",
            seed=SEED,
        )

        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=compute_metrics,
            class_weights=class_weights,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
        )
        trainer.train()
        metrics = trainer.evaluate()
        elapsed = time.time() - t0

        row = {
            "run_name": cfg.name,
            "model_name": cfg.model_name,
            "learning_rate": cfg.learning_rate,
            "num_train_epochs": cfg.num_train_epochs,
            "train_batch_size": cfg.train_batch_size,
            "weight_decay": cfg.weight_decay,
            "max_length": cfg.max_length,
            "eval_accuracy": metrics.get("eval_accuracy"),
            "eval_macro_f1": metrics.get("eval_macro_f1"),
            "train_seconds": elapsed,
        }
        all_results.append(row)
        if best is None or (row["eval_macro_f1"] > best["eval_macro_f1"]):
            best = {**row, "trainer": trainer, "tokenizer": tokenizer}
    except Exception as e:
        print(f"[skip] {cfg.name} failed: {type(e).__name__}: {e}")

if not all_results:
    raise RuntimeError("All configs failed. Check environment and model downloads.")

results_df = pd.DataFrame(all_results).sort_values("eval_macro_f1", ascending=False)
results_df


=== Training: roberta_base_lr2e5_ep4_len384 (roberta-base) on mps ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6752.84it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 2221/2221 [00:00<00:00, 4534.51 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in 

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.743174,0.699972,0.665016,0.666215
2,0.639901,0.717282,0.660513,0.660523


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.639901,0.699972,2,0.665016,0.666215


,run_name,model_name,learning_rate,num_train_epochs,train_batch_size,weight_decay,max_length,eval_accuracy,eval_macro_f1,train_seconds
0,roberta_base_lr2e5_ep4_len384,roberta-base,0.00002,4,16,0.01,384,0.665016,0.666215,2452.178378


In [12]:
results_path = ARTIFACT_DIR / "config_results.csv"
results_df.to_csv(results_path, index=False)
print("Saved config results:", results_path)

best_dir = ARTIFACT_DIR / "best_model"
best_dir.mkdir(parents=True, exist_ok=True)
best["trainer"].save_model(str(best_dir))
best["tokenizer"].save_pretrained(str(best_dir))

best_meta = {k: v for k, v in best.items() if k not in {"trainer", "tokenizer"}}
with open(ARTIFACT_DIR / "best_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(best_meta, f, indent=2)

print("Best run:", best_meta)
print("Saved model to:", best_dir)

Saved config results: /Users/aryanchoudhary/VSCodeProjects/COMP_488/488_Capstone_Text_Classifier/specialist_model_artifacts/config_results.csv


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

Best run: {'run_name': 'roberta_base_lr2e5_ep4_len384', 'model_name': 'roberta-base', 'learning_rate': 2e-05, 'num_train_epochs': 4, 'train_batch_size': 16, 'weight_decay': 0.01, 'max_length': 384, 'eval_accuracy': 0.665015758667267, 'eval_macro_f1': 0.6662153557683744, 'train_seconds': 2452.178377866745}
Saved model to: /Users/aryanchoudhary/VSCodeProjects/COMP_488/488_Capstone_Text_Classifier/specialist_model_artifacts/best_model


## Holdout evaluation (run only after selecting best config)

In [13]:
holdout_text = pd.read_csv(HOLDOUT_TEXT_CSV)[["complaint_id", "complaint_what_happened"]].copy()
holdout_truth = pd.read_csv(HOLDOUT_TRUTH_CSV)[["complaint_id", "consensus_category_slugs"]].copy()
holdout_truth = holdout_truth.rename(columns={"consensus_category_slugs": "y_true"})
holdout_truth["y_true"] = holdout_truth["y_true"].map(normalize_label)
holdout_truth = holdout_truth[holdout_truth["y_true"].isin(LABELS)]

h = holdout_text.merge(holdout_truth, on="complaint_id", how="inner")
h["text"] = h["complaint_what_happened"].fillna("").astype(str)
h = h[h["text"].str.len() > 0].reset_index(drop=True)

best_model = AutoModelForSequenceClassification.from_pretrained(str(best_dir)).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(str(best_dir))
best_model.eval()

# Use the same max_length the model was trained with so we don't truncate
# 50%+ of holdout texts at inference and tank macro-F1.
infer_max_length = int(best_meta.get("max_length", 384))
batch_size = 32 if DEVICE != "cpu" else 16
pred_ids = []
start = time.time()
for i in range(0, len(h), batch_size):
    texts = h["text"].iloc[i:i+batch_size].tolist()
    enc = best_tokenizer(texts, truncation=True, max_length=infer_max_length, padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = best_model(**enc).logits
    pred_ids.extend(torch.argmax(logits, dim=-1).cpu().tolist())
infer_seconds = time.time() - start

h["y_pred"] = [ID2LABEL[i] for i in pred_ids]

specialist_metrics = {
    "model": "specialist_best",
    "n": len(h),
    "accuracy": accuracy_score(h["y_true"], h["y_pred"]),
    "macro_f1": f1_score(h["y_true"], h["y_pred"], average="macro"),
    "inference_seconds": infer_seconds,
    "texts_per_second": len(h) / max(infer_seconds, 1e-9),
}
print(specialist_metrics)
print(classification_report(h["y_true"], h["y_pred"], labels=LABELS, digits=3))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13656.47it/s]


{'model': 'specialist_best', 'n': 100, 'accuracy': 0.81, 'macro_f1': 0.810224387566047, 'inference_seconds': 2.7935233116149902, 'texts_per_second': 35.79708806589054}
                          precision    recall  f1-score   support

        improper_charges      0.824     0.903     0.862        31
        improper_process      0.732     0.857     0.789        35
deceptive_discriminatory      0.920     0.676     0.780        34

                accuracy                          0.810       100
               macro avg      0.825     0.812     0.810       100
            weighted avg      0.824     0.810     0.808       100



In [14]:
def score_genai_file(path: Path, truth_df: pd.DataFrame):
    d = pd.read_csv(path)
    d["complaint_category"] = d["complaint_category"].map(normalize_label)
    d = d[d["complaint_category"].isin(LABELS)].copy()
    d = d[["complaint_id", "complaint_category", "created_at"]].rename(columns={"complaint_category": "y_pred"})
    m = truth_df.merge(d, on="complaint_id", how="inner")
    if m.empty:
        return None
    duration = np.nan
    if "created_at" in m.columns and m["created_at"].notna().any():
        ts = pd.to_datetime(m["created_at"], errors="coerce", utc=True).dropna()
        if len(ts) >= 2:
            duration = (ts.max() - ts.min()).total_seconds()
    return {
        "model": path.parent.name,
        "n": len(m),
        "accuracy": accuracy_score(m["y_true"], m["y_pred"]),
        "macro_f1": f1_score(m["y_true"], m["y_pred"], average="macro"),
        "elapsed_seconds": duration,
        "texts_per_second": (len(m) / duration) if pd.notna(duration) and duration > 0 else np.nan,
    }

truth_for_genai = holdout_truth[["complaint_id", "y_true"]].copy()
rows = []
for p in sorted(GEN_AI_OUTPUT_DIR.glob("*/labels_remapped.csv")):
    s = score_genai_file(p, truth_for_genai)
    if s:
        rows.append(s)

genai_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
genai_df.head(10)

,model,n,accuracy,macro_f1,elapsed_seconds,texts_per_second
3,anthropic__claude-sonnet-4-6_native,100,0.820000,0.819591,3174.995136,0.031496
1,anthropic__claude-opus-4-7,100,0.650000,0.641176,179410.743396,0.000557
7,google__gemini-2.5-flash,100,0.660000,0.631381,282.475472,0.354013
2,anthropic__claude-sonnet-4-6,100,0.650000,0.625248,63269.197118,0.001581
0,anthropic__claude-haiku-4-5,99,0.616162,0.570298,164101.952195,0.000603
9,google__gemini-2.5-pro,100,0.590000,0.535242,93740.034676,0.001067
5,deepseek__chat-temp03,100,0.520000,0.482872,62.431325,1.601760
8,google__gemini-2.5-flash-lite,100,0.540000,0.476174,99910.852178,0.001001
4,deepseek__chat-temp0,100,0.500000,0.472222,176.303819,0.567203
6,deepseek__chat-temp07,100,0.480000,0.453329,61.965498,1.613801


In [15]:
best_genai = genai_df.iloc[0].to_dict()
comparison = pd.DataFrame([
    {
        "system": "specialist_best",
        "accuracy": specialist_metrics["accuracy"],
        "macro_f1": specialist_metrics["macro_f1"],
        "texts_per_second": specialist_metrics["texts_per_second"],
    },
    {
        "system": best_genai["model"],
        "accuracy": best_genai["accuracy"],
        "macro_f1": best_genai["macro_f1"],
        "texts_per_second": best_genai.get("texts_per_second", np.nan),
    }
])

comparison["macro_f1_delta_vs_genai"] = comparison["macro_f1"] - float(best_genai["macro_f1"])
comparison["accuracy_delta_vs_genai"] = comparison["accuracy"] - float(best_genai["accuracy"])
comparison


,system,accuracy,macro_f1,texts_per_second,macro_f1_delta_vs_genai,accuracy_delta_vs_genai
0,specialist_best,0.81,0.810224,35.797088,-0.009366,-0.01
1,anthropic__claude-sonnet-4-6_native,0.82,0.819591,0.031496,0.000000,0.00


## Cost and reproducibility section (Step 10)

- **Cheaper per 10,000 texts**: add your real API prices and GPU/CPU runtime cost assumptions below.
- **Faster**: compare `texts_per_second` from specialist vs best GenAI row.
- **Consistency/reproducibility**: specialist is deterministic under fixed seed/checkpoint; API models can drift over time and may vary by provider updates.

In [17]:
# Fill these values with your actual pricing assumptions.
# Example units:
# - specialist_compute_cost_per_hour: dollars/hour of your training+inference hardware
# - genai_cost_per_1k_texts: direct API cost for 1,000 complaints

pricing = {
    "specialist_compute_cost_per_hour": 0,
    "genai_cost_per_1k_texts": 3.01,
}

specialist_cost_per_10k = np.nan
if pd.notna(pricing["specialist_compute_cost_per_hour"]):
    total_hours = (best["train_seconds"] + specialist_metrics["inference_seconds"]) / 3600
    # spread one-time training cost over 10k for this assignment's comparison
    specialist_cost_per_10k = pricing["specialist_compute_cost_per_hour"] * total_hours * (10000 / max(specialist_metrics["n"], 1))

genai_cost_per_10k = np.nan
if pd.notna(pricing["genai_cost_per_1k_texts"]):
    genai_cost_per_10k = pricing["genai_cost_per_1k_texts"] * 10

pd.DataFrame([
    {"system": "specialist_best", "estimated_cost_per_10k": specialist_cost_per_10k},
    {"system": str(best_genai["model"]), "estimated_cost_per_10k": genai_cost_per_10k},
])

,system,estimated_cost_per_10k
0,specialist_best,0.0
1,anthropic__claude-sonnet-4-6_native,30.1
